# Step 0 - Set file path
Read the CSV file generated by the "Extract Data" Notebook for individuals aged 60 and above. If the file location is different, only modify the "folder". This Notebook will not modify the "Extract Data" Notebook.

In [21]:
from pathlib import Path
import pandas as pd

folder = Path("/Users/densoul/Desktop/600")
input_file = folder / "hrs_full_data.csv"
output_file = folder / "hrs_data_cleaning_first.csv"

if not input_file.is_file():
    raise FileNotFoundError(f"Input file not found: {input_file}")

years = (2016, 2018, 2020, 2022)
score_col = "Derived 0-27 cognition score"

component_limits = {
    "Immediate recall": 10,
    "Delayed recall": 10,
    "Serial 7s": 5,
    "Backwards counting": 2,
}

disease_cols = [
    "Hypertension", "Diabetes", "Cancer", "Chronic lung disease",
    "Heart disease", "Stroke", "Arthritis",
]

# Step 1 - Read and verify the data structure
Confirm that the file contains four survey years, respondent IDs, ages, cognitive scores, and seven diseases. Check for any duplicate entries for the same person in the same year and again ensure that the age is at least 60 years old. This step does not modify the cognitive scores.

In [22]:
data = pd.read_csv(
    input_file,
    dtype={"Participant ID": "string"},
    low_memory=False,
)
data.columns = data.columns.str.strip()

required_cols = {
    "Participant ID", "Survey year", "Age", score_col,
    "Sex", "Race/ethnicity", "Education", "Marital status",
    "Smoking", "BMI", "Interview mode", "Proxy respondent",
    *component_limits.keys(), *disease_cols,
}
missing_cols = sorted(required_cols - set(data.columns))
if missing_cols:
    raise ValueError(f"Required columns are missing: {missing_cols}")

data["Participant ID"] = data["Participant ID"].str.strip()
data["Survey year"] = pd.to_numeric(data["Survey year"], errors="coerce")
data["Age"] = pd.to_numeric(data["Age"], errors="coerce")
data[score_col] = pd.to_numeric(data[score_col], errors="coerce")

if data["Participant ID"].isna().any() or data["Participant ID"].eq("").any():
    raise ValueError("Missing participant IDs found.")
if data[["Participant ID", "Survey year"]].duplicated().any():
    raise ValueError("Duplicate participant-year records found.")
if data["Survey year"].isna().any() or not data["Survey year"].isin(years).all():
    raise ValueError("Unexpected or missing survey year found.")

summary = data.groupby("Survey year").agg(
    records=("Participant ID", "size"),
    missing_age=("Age", lambda s: int(s.isna().sum())),
    age_65plus=("Age", lambda s: int(s.ge(65).sum())),
    missing_cognition=(score_col, lambda s: int(s.isna().sum())),
)

print(f"Input participant-year records: {len(data):,}")
print(summary.to_string())

Input participant-year records: 180,936
             records  missing_age  age_65plus  missing_cognition
Survey year                                                     
2016           45234        24322        9985              25263
2018           45234        28088        9012              28751
2020           45234        29511        8999              30232
2022           45234        29378        8995              30041


# Step 2 — Restrict the sample to adults aged 65 or older

Keep participant-year records with a valid age of 65 or above. Exclude records with age below 65 or missing age, and report each exclusion separately.

The output shows the number of records before and after filtering, the number of unique participants retained, and the retained records for each survey year. No other features are used to exclude records in this step.

In [23]:
data["Age"] = pd.to_numeric(data["Age"], errors="coerce")

records_before = len(data)
under_65 = data["Age"].notna() & data["Age"].lt(65)
missing_age = data["Age"].isna()
age_65plus = data["Age"].ge(65)

data = data.loc[age_65plus].copy()

print(f"Records before age filter: {records_before:,}")
print(f"Records under 65 excluded: {int(under_65.sum()):,}")
print(f"Records with missing age excluded: {int(missing_age.sum()):,}")
print(f"Total participant-year records aged 65+: {len(data):,}")
print(f"Unique participants aged 65+: {data['Participant ID'].nunique():,}")
print("Records by survey year:")
print(data.groupby("Survey year").size().to_string())

Records before age filter: 180,936
Records under 65 excluded: 32,646
Records with missing age excluded: 111,299
Total participant-year records aged 65+: 36,991
Unique participants aged 65+: 14,380
Records by survey year:
Survey year
2016    9985
2018    9012
2020    8999
2022    8995


# Step 3 — Remove missing and invalid records

Check missing values and invalid codes in all required features before removing records. Required features are the cognition score and its four components, seven disease indicators, the existing chronic disease count, age, sex, race/ethnicity, education, marital status, smoking, BMI, and proxy respondent status.

The chronic disease count was calculated in the Extract Data notebook. Verify that it equals the sum of the seven valid 0/1 disease indicators. After removing invalid records, rename this column to `Chronic Disease Burden` without recalculating it.

Keep only self-respondents (`Proxy respondent = 0`). Do not exclude records because Interview mode is missing; this field is missing by design in 2016. The official cognition score is not required to equal the simple sum of its four components because web interview scores may be mode-adjusted.

Report issue counts before deletion. A participant may still be retained in another year if that year's record is valid.

In [24]:
import numpy as np

required_features = [
    score_col,
    *component_limits.keys(),
    *disease_cols,
    "Chronic disease count",
    "Age",
    "Sex",
    "Race/ethnicity",
    "Education",
    "Marital status",
    "Smoking",
    "BMI",
    "Proxy respondent",
]

missing_columns = sorted(set(required_features) - set(data.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

numeric_features = [
    score_col,
    *component_limits.keys(),
    *disease_cols,
    "Chronic disease count",
    "Age",
    "Sex",
    "Education",
    "Marital status",
    "Smoking",
    "BMI",
    "Proxy respondent",
]

for name in numeric_features:
    data[name] = pd.to_numeric(data[name], errors="coerce")

data["Race/ethnicity"] = (
    data["Race/ethnicity"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

issues = pd.DataFrame(index=data.index)

issues["missing_required"] = (
    data[required_features].isna().any(axis=1)
)

issues["age_out_of_range"] = (
    data["Age"].notna()
    & ~data["Age"].between(65, 120)
)

issues["cognition_score_out_of_range"] = (
    data[score_col].notna()
    & ~data[score_col].between(0, 27)
)

component_invalid = pd.Series(False, index=data.index)
for name, maximum in component_limits.items():
    component_invalid |= (
        data[name].notna()
        & ~data[name].between(0, maximum)
    )
issues["cognition_component_invalid"] = component_invalid

disease_invalid = pd.DataFrame({
    name: data[name].notna() & ~data[name].isin([0, 1])
    for name in disease_cols
}, index=data.index)
issues["disease_code_invalid"] = disease_invalid.any(axis=1)

recalculated_count = (
    data[disease_cols]
    .where(data[disease_cols].isin([0, 1]))
    .sum(axis=1, min_count=7)
)

issues["disease_count_out_of_range"] = (
    data["Chronic disease count"].notna()
    & ~data["Chronic disease count"].between(0, 7)
)

issues["disease_count_mismatch"] = (
    recalculated_count.notna()
    & data["Chronic disease count"].notna()
    & recalculated_count.ne(data["Chronic disease count"])
)

issues["sex_invalid"] = (
    data["Sex"].notna()
    & ~data["Sex"].isin([1, 2])
)

valid_race_groups = {
    "Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Other",
}
issues["race_ethnicity_invalid"] = (
    data["Race/ethnicity"].notna()
    & ~data["Race/ethnicity"].isin(valid_race_groups)
)

issues["education_invalid"] = (
    data["Education"].notna()
    & ~data["Education"].between(0, 17)
)

issues["marital_status_invalid"] = (
    data["Marital status"].notna()
    & ~data["Marital status"].isin(range(1, 9))
)

issues["smoking_invalid"] = (
    data["Smoking"].notna()
    & ~data["Smoking"].isin([0, 1])
)

issues["bmi_invalid"] = (
    data["BMI"].notna()
    & (
        data["BMI"].le(0)
        | ~np.isfinite(data["BMI"])
    )
)

issues["proxy_not_self"] = (
    data["Proxy respondent"].notna()
    & data["Proxy respondent"].ne(0)
)

issues = issues.fillna(False).astype(bool)
exclude_mask = issues.any(axis=1)

print("Missing values by required feature:")
print(
    data[required_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

print("Invalid disease codes by condition:")
print(disease_invalid.sum().astype(int).to_string())

print("Issue counts (categories may overlap):")
print(
    issues.sum()
    .astype(int)
    .sort_values(ascending=False)
    .to_string()
)

print(f"Records before removal: {len(data):,}")
print(f"Unique records flagged for removal: {int(exclude_mask.sum()):,}")

by_year = (
    data.assign(flagged_for_removal=exclude_mask)
    .groupby("Survey year")
    .agg(
        records_before=("Participant ID", "size"),
        records_flagged=("flagged_for_removal", "sum"),
    )
)
by_year["records_remaining"] = (
    by_year["records_before"] - by_year["records_flagged"]
)

print("Record counts by survey year:")
print(by_year.to_string())

Missing values by required feature:
Derived 0-27 cognition score    2320
Delayed recall                  2320
Serial 7s                       2320
Backwards counting              2320
Immediate recall                2320
Smoking                         1746
BMI                              484
Education                        133
Marital status                    65
Race/ethnicity                    44
Age                                0
Sex                                0
Stroke                             0
Chronic disease count              0
Arthritis                          0
Heart disease                      0
Chronic lung disease               0
Cancer                             0
Diabetes                           0
Hypertension                       0
Proxy respondent                   0
Invalid disease codes by condition:
Hypertension            0
Diabetes                0
Cancer                  0
Chronic lung disease    0
Heart disease           0
Stroke               

In [25]:
records_before = len(data)
data = data.loc[~exclude_mask].copy()

data = data.rename(columns={
    "Chronic disease count": "Chronic Disease Burden"
})

assert data["Chronic Disease Burden"].between(0, 7).all()
assert data["Proxy respondent"].eq(0).all()

print(f"Records removed: {records_before - len(data):,}")
print(f"Participant-year records remaining: {len(data):,}")
print(f"Unique participants remaining: {data['Participant ID'].nunique():,}")
print("Remaining records by survey year:")
print(data.groupby("Survey year").size().to_string())

Records removed: 4,642
Participant-year records remaining: 32,349
Unique participants remaining: 13,061
Remaining records by survey year:
Survey year
2016    9042
2018    8306
2020    8222
2022    6779


# Step 4 - Review extreme BMI values

Read RAND height and weight for the same participant and survey year. Flag BMI values below 12 or above 60 for review; these are project screening thresholds, not official HRS invalid-value codes. Compare the reported BMI with weight divided by height squared and inspect the source measurements. A matching calculation does not establish that self-reported height or weight is accurate. Remove a participant-year record only after deciding that its BMI is unreliable and cannot be corrected.

In [26]:
from pathlib import Path
import pandas as pd
import numpy as np

rand_file = folder / "randhrs1992_2022v1_STATA" / "randhrs1992_2022v1.dta"
if not rand_file.is_file():
    raise FileNotFoundError(f"RAND file not found: {rand_file}")

waves = {13: 2016, 14: 2018, 15: 2020, 16: 2022}
source_cols = ["hhid", "pn"]
for wave in waves:
    source_cols.extend([f"r{wave}height", f"r{wave}weight"])

rand_hw = pd.read_stata(
    rand_file,
    columns=source_cols,
    convert_categoricals=False,
)

def normalize_id(series, width):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(width)
    )

rand_hw["Participant ID"] = (
    normalize_id(rand_hw["hhid"], 6)
    + normalize_id(rand_hw["pn"], 3)
)

parts = []
for wave, year in waves.items():
    part = rand_hw[
        ["Participant ID", f"r{wave}height", f"r{wave}weight"]
    ].rename(
        columns={
            f"r{wave}height": "Height (m)",
            f"r{wave}weight": "Weight (kg)",
        }
    ).copy()
    part["Survey year"] = year
    parts.append(part)

height_weight = pd.concat(parts, ignore_index=True)

if height_weight.duplicated(["Participant ID", "Survey year"]).any():
    raise ValueError("Duplicate participant-year records in RAND height and weight.")

bmi_check = data[
    ["Participant ID", "Survey year", "BMI"]
].merge(
    height_weight,
    on=["Participant ID", "Survey year"],
    how="left",
    validate="one_to_one",
)

for column in ["BMI", "Height (m)", "Weight (kg)"]:
    bmi_check[column] = pd.to_numeric(
        bmi_check[column], errors="coerce"
    ).astype(float)

print(f"Records available for BMI review: {len(bmi_check):,}")

Records available for BMI review: 32,349


In [28]:
height = bmi_check["Height (m)"]
weight = bmi_check["Weight (kg)"]
bmi = bmi_check["BMI"]

bmi_exclusion_rules = pd.DataFrame(
    {
        "height_outside_broad_range": height.notna()
        & (height.lt(1.00) | height.gt(2.70)),

        "weight_outside_broad_range": weight.notna()
        & (weight.lt(20) | weight.gt(350)),

        "bmi_outside_broad_range": bmi.notna()
        & (bmi.lt(11) | bmi.gt(75)),

        "invalid_height_or_weight": (
            (height.notna() & (~np.isfinite(height) | height.le(0)))
            | (weight.notna() & (~np.isfinite(weight) | weight.le(0)))
        ),

        "bmi_calculation_differs_over_10_percent": (
            bmi_check["Relative difference"].gt(0.10).fillna(False)
        ),
    },
    index=bmi_check.index,
)

bmi_check["Exclude BMI"] = bmi_exclusion_rules.any(axis=1)

bmi_check["Review only"] = (
    ~bmi_check["Exclude BMI"]
    & (
        height.lt(1.20) | height.gt(2.10)
        | weight.lt(35) | weight.gt(180)
        | bmi.lt(12) | bmi.gt(60)
        | bmi_check["Relative difference"].gt(0.05).fillna(False)
    )
)

print("Exclusion rules:")
print("Height outside 1.00-2.70 m")
print("Weight outside 20-350 kg")
print("BMI outside 11-75")
print("Nonpositive or nonfinite height or weight")
print("BMI calculation difference above 10%")

print("\nNumber of records meeting each exclusion rule:")
print(bmi_exclusion_rules.sum().astype(int).to_string())

decision_summary = bmi_check.groupby("Survey year").agg(
    input_records=("BMI", "size"),
    exclude_bmi=("Exclude BMI", "sum"),
    review_only=("Review only", "sum"),
)
decision_summary["keep"] = (
    decision_summary["input_records"] - decision_summary["exclude_bmi"]
)

print("\nDecision summary by year:")
print(decision_summary.to_string())
print(f"\nTotal records selected for deletion: {int(bmi_check['Exclude BMI'].sum()):,}")

print("\nRecords selected for deletion:")
display(
    bmi_check.loc[
        bmi_check["Exclude BMI"],
        [
            "Participant ID", "Survey year", "BMI",
            "Height (m)", "Weight (kg)",
            "Recalculated BMI", "Relative difference",
        ],
    ].sort_values("BMI")
)

Exclusion rules:
Height outside 1.00-2.70 m
Weight outside 20-350 kg
BMI outside 11-75
Nonpositive or nonfinite height or weight
BMI calculation difference above 10%

Number of records meeting each exclusion rule:
height_outside_broad_range                  2
weight_outside_broad_range                  0
bmi_outside_broad_range                    20
invalid_height_or_weight                    0
bmi_calculation_differs_over_10_percent     0

Decision summary by year:
             input_records  exclude_bmi  review_only  keep
Survey year                                               
2016                  9042            2            6  9040
2018                  8306            1           13  8305
2020                  8222            9           12  8213
2022                  6779            8           12  6771

Total records selected for deletion: 20

Records selected for deletion:


,Participant ID,Survey year,BMI,Height (m),Weight (kg),Recalculated BMI,Relative difference
22818,500661010,2020,8.9,1.67640,24.94745,8.877095,0.002574
20765,087659020,2020,9.1,1.73990,27.66899,9.139969,0.004392
30093,502710010,2022,9.4,1.54940,22.67950,9.447271,0.005029
20067,074220010,2020,9.4,1.54940,22.67950,9.447271,0.005029
32281,918887020,2022,9.4,1.62560,24.94745,9.440582,0.004317
29108,500661020,2022,9.8,1.77800,30.84412,9.756833,0.004405
32011,910179010,2022,10.0,1.65100,27.21540,9.984368,0.001563
20764,087659010,2020,10.2,1.70180,29.48335,10.180282,0.001933
19669,056778010,2020,10.3,1.68275,29.02976,10.251897,0.004670
7359,500046010,2016,10.3,1.62560,27.21540,10.298817,0.000115


In [29]:
bmi_decisions = bmi_check[
    ["Participant ID", "Survey year", "Exclude BMI"]
].copy()

if bmi_decisions.duplicated(["Participant ID", "Survey year"]).any():
    raise ValueError("Duplicate BMI decisions for a participant-year record.")

data_with_decision = data.merge(
    bmi_decisions,
    on=["Participant ID", "Survey year"],
    how="left",
    validate="one_to_one",
)

if data_with_decision["Exclude BMI"].isna().any():
    raise ValueError("Some records do not have a BMI cleaning decision.")

flow = data_with_decision.groupby("Survey year")["Exclude BMI"].agg(
    input_records="size",
    deleted_records="sum",
)
flow["remaining_records"] = (
    flow["input_records"] - flow["deleted_records"]
)

data = (
    data_with_decision.loc[~data_with_decision["Exclude BMI"]]
    .drop(columns="Exclude BMI")
    .copy()
)

print("BMI cleaning results by year:")
print(flow.to_string())
print(f"\nTotal records after BMI cleaning: {len(data):,}")

BMI cleaning results by year:
             input_records  deleted_records  remaining_records
Survey year                                                   
2016                  9042                2               9040
2018                  8306                1               8305
2020                  8222                9               8213
2022                  6779                8               6771

Total records after BMI cleaning: 32,329


# Step 5 — Keep the latest valid year for each participant

After the required data-quality checks, retain one record per participant. If a participant has records from multiple survey years, keep the most recent remaining year and remove earlier years.

Report how many participants had multiple records, how many earlier-year records were removed, and the final number of participants and records.

In [30]:
records_before = len(data)
participants_before = data["Participant ID"].nunique()
participants_with_multiple_years = int(
    data.groupby("Participant ID").size().gt(1).sum()
)

data = (
    data.sort_values(
        ["Participant ID", "Survey year"],
        ascending=[True, False],
        kind="stable",
    )
    .drop_duplicates(subset="Participant ID", keep="first")
    .reset_index(drop=True)
)

assert data["Participant ID"].is_unique
assert len(data) == participants_before

print("Participants with multiple years:", participants_with_multiple_years)
print("Earlier-year records removed:", records_before - len(data))
print(f"Final participant records: {len(data):,}")
print("Final records by survey year:")
print(data.groupby("Survey year").size().to_string())

Participants with multiple years: 9234
Earlier-year records removed: 19276
Final participant records: 13,053
Final records by survey year:
Survey year
2016    1578
2018    1486
2020    3218
2022    6771


# Step 6- Save the csv

In [32]:
final_file = folder / "cleaned_latest_data.csv"
data.to_csv(final_file, index=False, encoding="utf-8-sig")

print(f"Saved: {final_file}")
print(f"Final records: {len(data):,}")
print(f"Unique participants: {data['Participant ID'].nunique():,}")

Saved: /Users/densoul/Desktop/600/cleaned_latest_data.csv
Final records: 13,053
Unique participants: 13,053
